In [53]:
#!/usr/bin/env python3
import sys
import os
import glob
import re
import cv2
import numpy as np
import matplotlib
matplotlib.use('TkAgg')  # Try this backend for displaying plots
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.widgets import Slider, Button
from tqdm import tqdm  # For progress bars (install with pip install tqdm if needed)
import transforms3d.euler as euler


In [ ]:
file_path = '/home/ranger/04_13_2025_01_55_41_trial_1/np_data_04_13_2025_01_55_41.npz'
gen_fpath = '/home/ranger/04_13_2025_01_55_41_trial_1'
# Load the NPZ file
npz_file = np.load(file_path, allow_pickle=True)
keys = list(npz_file.keys())
print(f"Keys found: {keys}")

# Create a directory to save plots
plot_dir = os.path.join(os.path.dirname(file_path), "plots")
os.makedirs(plot_dir, exist_ok=True)
print(f"Saving plots to: {plot_dir}")

Keys found: ['q', 'dq', 't', 'input', 'u', 'qd', 'dqd', 'depth', 'depth_d', 'quat', 'alt', 'yaw_d', 'stereo_depth', 'stereo_point']
Saving plots to: /home/ranger/04_13_2025_01_55_41_trial_1/plots


### Motor Positions and Desired Positions (q and dq)

In [55]:
print(npz_file['t'][:50])

[0.17005014 0.17405868 0.17723918 0.18006945 0.18305874 0.18604183
 0.19016981 0.19417739 0.19814014 0.20315886 0.20715451 0.21114802
 0.21514082 0.21916151 0.22413659 0.22815537 0.23347759 0.24019098
 0.24416208 0.24914646 0.25515366 0.26244497 0.27153063 0.27813697
 0.28424883 0.29142237 0.29666948 0.30104113 0.30406356 0.30806828
 0.31304502 0.31603026 0.31920767 0.32405901 0.32816291 0.33204246
 0.33504009 0.33805132 0.34204173 0.34603381 0.34904957 0.35306835
 0.35605288 0.36004996 0.36305213 0.36603761 0.36924696 0.3720386
 0.37603736 0.37904501]


In [5]:
# Get the data
q_data = npz_file['q']
qd_data = npz_file['qd']

print(f"q data shape: {q_data.shape}")
print(f"qd data shape: {qd_data.shape}")
    
# Define range to plot
start_idx = 0
end_idx = q_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(q_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

# Create individual plots for each dimension
for i in range(10):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Plot both q and dqd on same axis
    ax.plot(x, q_data[start_idx:end_idx, i], 'b-', label=f'q{i+1} (Position)', linewidth=1.5)
    ax.plot(x, qd_data[start_idx:end_idx, i], 'r-', label=f'qd{i+1} (Desired Position)', linewidth=1.5)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel('radians')
    ax.set_title(f'Dimension {i+1}: Position and Desired Position')
    
    # Add legend
    ax.legend(loc='best')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits to ensure both datasets are visible
    y_min = min(np.min(q_data[start_idx:end_idx, i]), np.min(qd_data[start_idx:end_idx, i]))
    y_max = max(np.max(q_data[start_idx:end_idx, i]), np.max(qd_data[start_idx:end_idx, i]))
    
    # Add some padding to the y-axis limits
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(plot_dir, f"position_motor_{i+1}.png")
    plt.savefig(save_path, dpi=300)
    print(f"Saved plot for dimension {i+1} to {save_path}")
    
    # Close the plot to free memory
    plt.close(fig)

# Create a combined figure with all dimensions (5x2 grid)
fig, axes = plt.subplots(5, 2, figsize=(18, 24))
axes = axes.flatten()

for i in range(10):
    ax = axes[i]
    
    # Plot both q and dqd on same axis
    ax.plot(x, q_data[start_idx:end_idx, i], 'b-', label=f'q{i+1}', linewidth=1.2)
    ax.plot(x, qd_data[start_idx:end_idx, i], 'r-', label=f'qd{i+1}', linewidth=1.2)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel('radians')
    ax.set_title(f'Motor {i+1}')
    
    # Add legend
    ax.legend(loc='best', fontsize='small')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits
    y_min = min(np.min(q_data[start_idx:end_idx, i]), np.min(qd_data[start_idx:end_idx, i]))
    y_max = max(np.max(q_data[start_idx:end_idx, i]), np.max(qd_data[start_idx:end_idx, i]))
    
    # Add padding
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)

plt.tight_layout()
# Save the combined plot
combined_path = os.path.join(plot_dir, "all_positions.png")
plt.savefig(combined_path, dpi=300)
print(f"Saved plot to {combined_path}")
plt.close()


q data shape: (40297, 10)
qd data shape: (40297, 10)
Saved plot for dimension 1 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_1.png
Saved plot for dimension 2 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_2.png
Saved plot for dimension 3 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_3.png
Saved plot for dimension 4 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_4.png
Saved plot for dimension 5 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_5.png
Saved plot for dimension 6 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_6.png
Saved plot for dimension 7 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_7.png
Saved plot for dimension 8 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_8.png
Saved plot for dimension 9 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/position_motor_9.png
Saved plot for dimension 10 to /home/ranger/04_13_2025_01_55_41_t

### Velocities and Desired Velocities (dq and dqd)

In [6]:
# Get the data
dq_data = npz_file['dq']
dqd_data = npz_file['dqd']

print(f"dq data shape: {dq_data.shape}")
print(f"dqd data shape: {dqd_data.shape}")
    
# Define range to plot
start_idx = 0
end_idx = dq_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(q_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

# Create individual plots for each dimension
for i in range(10):
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Plot both q and dqd on same axis
    ax.plot(x, dq_data[start_idx:end_idx, i], 'b-', label=f'dq{i+1} (Velocity)', linewidth=1.5)
    ax.plot(x, dqd_data[start_idx:end_idx, i], 'r-', label=f'dqd{i+1} (Desired Velocity)', linewidth=1.5)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel(r'rad $\cdot$ sec$^{-1}$')
    ax.set_title(f'Motor {i+1}: Velocity and Desired Velocity')
    
    # Add legend
    ax.legend(loc='best')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits to ensure both datasets are visible
    y_min = min(np.min(dq_data[start_idx:end_idx, i]), np.min(dqd_data[start_idx:end_idx, i]))
    y_max = max(np.max(dq_data[start_idx:end_idx, i]), np.max(dqd_data[start_idx:end_idx, i]))
    
    # Add some padding to the y-axis limits
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.join(plot_dir, f"velocity_motor_{i+1}.png")
    plt.savefig(save_path, dpi=300)
    print(f"Saved plot for dimension {i+1} to {save_path}")
    
    # Close the plot to free memory
    plt.close(fig)

# Create a combined figure with all dimensions (5x2 grid)
fig, axes = plt.subplots(5, 2, figsize=(18, 24))
axes = axes.flatten()

for i in range(10):
    ax = axes[i]
    
    # Plot both q and dqd on same axis
    ax.plot(x, q_data[start_idx:end_idx, i], 'b-', label=f'dq{i+1}', linewidth=1.2)
    ax.plot(x, qd_data[start_idx:end_idx, i], 'r-', label=f'dqd{i+1}', linewidth=1.2)
    
    # Add labels and title
    ax.set_xlabel(x_label)
    ax.set_ylabel(r'rad $\cdot$ sec$^{-1}$')
    ax.set_title(f'Motor {i+1}')
    
    # Add legend
    ax.legend(loc='best', fontsize='small')
    
    # Add grid for readability
    ax.grid(True, alpha=0.3)
    
    # Calculate y-axis limits
    y_min = min(np.min(dq_data[start_idx:end_idx, i]), np.min(dqd_data[start_idx:end_idx, i]))
    y_max = max(np.max(dq_data[start_idx:end_idx, i]), np.max(dqd_data[start_idx:end_idx, i]))
    
    # Add padding
    y_range = y_max - y_min
    y_min -= 0.1 * y_range
    y_max += 0.1 * y_range
    
    ax.set_ylim(y_min, y_max)

plt.tight_layout()

# Save the combined plot
combined_path = os.path.join(plot_dir, "all_velocities.png")
plt.savefig(combined_path, dpi=300)
print(f"Saved plot to {combined_path}")
plt.close()


dq data shape: (40297, 10)
dqd data shape: (40297, 10)
Saved plot for dimension 1 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_1.png
Saved plot for dimension 2 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_2.png
Saved plot for dimension 3 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_3.png
Saved plot for dimension 4 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_4.png
Saved plot for dimension 5 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_5.png
Saved plot for dimension 6 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_6.png
Saved plot for dimension 7 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_7.png
Saved plot for dimension 8 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_8.png
Saved plot for dimension 9 to /home/ranger/04_13_2025_01_55_41_trial_1/plots/velocity_motor_9.png
Saved plot for dimension 10 to /home/ranger/04_13_2025_01_55_41

### Depth and Desired Depth

In [63]:
depth_data = npz_file['depth']
depth_d_data = npz_file['depth_d']
# Define range to plot
start_idx = 0
end_idx = depth_data.shape[0]

# Prepare x-axis data
if 't' in npz_file:
    t_data = npz_file['t']
    if len(t_data) == len(depth_data):
        x = t_data[start_idx:end_idx]
        x_label = 'Time (seconds)'
    else:
        x = np.arange(start_idx, end_idx)
        x_label = 'Index'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

fig, ax = plt.subplots(figsize=(12, 7))

# Plot both q and dqd on same axis
ax.plot(x, depth_data[start_idx:end_idx], 'b-', label='Depth', linewidth=1.5)
ax.plot(x, depth_d_data[start_idx:end_idx], 'r-', label='Desired Depth', linewidth=1.5)

# Add labels and title
ax.set_xlabel(x_label)
ax.set_ylabel('Depth (meters)')
ax.set_title('Depth and Desired Depth')

# Add legend
ax.legend(loc='best')

# Add grid for readability
ax.grid(True, alpha=0.3)

# Calculate y-axis limits to ensure both datasets are visible
y_min = min(np.min(depth_data[start_idx:end_idx]), np.min(depth_d_data[start_idx:end_idx]))
y_max = max(np.max(depth_data[start_idx:end_idx]), np.max(depth_d_data[start_idx:end_idx]))

# Add some padding to the y-axis limits
y_range = y_max - y_min
y_min -= 0.1 * y_range
y_max += 0.1 * y_range

ax.set_ylim(y_min, y_max)

plt.tight_layout()

# Save the plot
save_path = os.path.join(plot_dir, f"depth_data.png")
plt.savefig(save_path, dpi=300)
print(f"Saved plot for depth to {save_path}")

# Close the plot to free memory
plt.close(fig)


# Calculate errors
depth_error = depth_data[start_idx:end_idx] - depth_d_data[start_idx:end_idx]
rmse = np.sqrt(np.mean(np.square(depth_error)))
mae = np.mean(np.abs(depth_error))
max_error = np.max(np.abs(depth_error))

# Add error metrics to plot
error_text = f"RMSE: {rmse:.4f} m\nMAE: {mae:.4f} m\nMax Error: {max_error:.4f} m"
print(error_text)

Saved plot for depth to /home/ranger/04_13_2025_01_55_41_trial_1/plots/depth_data.png
RMSE: 0.4089 m
MAE: 0.3123 m
Max Error: 1.0100 m


### Stereo Depth Images

In [62]:
stereo_depth_data = npz_file['stereo_depth']

# Create a figure and get the axes object
fig, ax = plt.subplots(figsize=(12, 6))

if len(t_data) == len(stereo_depth_data):
    x = t_data[start_idx:end_idx]
    x_label = 'Time'
else:
    x = np.arange(start_idx, end_idx)
    x_label = 'Index'

# Plot using the axes object
ax.plot(x, stereo_depth_data[start_idx:end_idx])

# Add labels and title to the axes object
ax.set_xlabel(x_label)
ax.set_ylabel('Depth (meters)')
ax.set_title('Stereo Depth')
ax.grid(True)

plt.tight_layout()

# Save the plot
save_path = os.path.join(plot_dir, f"stereo_depth_plot.png")
plt.savefig(save_path, dpi=300)
print(f"Saved plot for stereo depth to {save_path}")
plt.close(fig)  # Close using the figure object

Saved plot for stereo depth to /home/ranger/04_13_2025_01_55_41_trial_1/plots/stereo_depth_plot.png


### Yaw Tracking 

In [61]:
quat = npz_file['quat']
yaw_d = npz_file['yaw_d']

yaw_d_resampled = yaw_d[::2]  # Take every other value
yaw_d = yaw_d_resampled
q_eul = np.zeros((len(quat), 3))  # Create array to store roll, pitch, yaw
for i in range(len(quat)):
    q_eul[i] = euler.quat2euler(quat[i], 'szyx')  # Convert each quaternion

# Extract yaw angles (third component)
yaw = q_eul[:, 2]
# Create a nice-looking plot
plt.figure(figsize=(14, 8))

# Check if time data is available
if 't' in npz_file and len(npz_file['t']) >= len(quat):
    # Use time as x-axis if available
    t = npz_file['t'][:len(quat)]
    plt.plot(t, yaw, 'b-', linewidth=2, label='Actual Yaw')
    plt.plot(t, yaw_d, 'r-', linewidth=2, label='Desired Yaw')
    plt.xlabel('Time')
else:
    # Otherwise use indices
    indices = np.arange(len(yaw))
    plt.plot(indices, yaw, 'b-', linewidth=2, label='Actual Yaw')
    plt.plot(indices, yaw_d, 'r-', linewidth=2, label='Desired Yaw')
    plt.xlabel('Sample Index')

# Convert radians to degrees if needed
# Uncomment these if your angles are in radians and you want to display in degrees
# yaw = np.degrees(yaw)
# yaw_d = np.degrees(yaw_d)

# Set labels and title
plt.ylabel('Yaw Angle')
plt.title('Comparison of Actual vs Desired Yaw Angle')

# Add grid and legend
plt.grid(True, alpha=0.3)
plt.legend(loc='best')

# Calculate y-axis limits to ensure both datasets are visible
y_min = min(np.min(yaw), np.min(yaw_d))
y_max = max(np.max(yaw), np.max(yaw_d))

# Add some padding to the y-axis limits
y_range = y_max - y_min
y_min -= 0.1 * y_range
y_max += 0.1 * y_range

plt.ylim(y_min, y_max)

# Tight layout for better appearance
plt.tight_layout()
# Save the plot
save_path = os.path.join(plot_dir, f"yaw_vs_yaw_desired.png")
# Save the plot if a path is provided
if save_path:
    plt.savefig(save_path, dpi=300)
    print(f"Plot saved to: {save_path}")

# Create an additional plot showing the error between actual and desired yaw
plt.figure(figsize=(14, 6))

# Calculate yaw error
yaw_error = yaw - yaw_d

# Plot the error
if 't' in npz_file and len(npz_file['t']) >= len(quat):
    t = npz_file['t'][:len(yaw_error)]
    plt.plot(t, yaw_error, 'g-', linewidth=2)
    plt.xlabel('Time')
else:
    plt.plot(indices, yaw_error, 'g-', linewidth=2)
    plt.xlabel('Sample Index')

plt.ylabel('Yaw Error (Actual - Desired)')
plt.title('Yaw Tracking Error')
plt.grid(True, alpha=0.3)

# Add a horizontal line at zero for reference
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()

# Save the error plot if a path is provided
if save_path:
    error_path = save_path.replace('.png', '_error.png')
    plt.savefig(error_path, dpi=300)
    print(f"Error plot saved to: {error_path}")

# Calculate and print some statistics about the tracking performance
mean_error = np.mean(yaw_error)
rmse = np.sqrt(np.mean(yaw_error**2))
max_error = np.max(np.abs(yaw_error))

print("\nYaw Tracking Performance:")
print(f"Mean Error: {mean_error:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Maximum Absolute Error: {max_error:.4f}")

# Show the plots
plt.show()



Plot saved to: /home/ranger/04_13_2025_01_55_41_trial_1/plots/yaw_vs_yaw_desired.png
Error plot saved to: /home/ranger/04_13_2025_01_55_41_trial_1/plots/yaw_vs_yaw_desired_error.png

Yaw Tracking Performance:
Mean Error: 0.2620
RMSE: 0.9820
Maximum Absolute Error: 3.2516


### Depth Images Recording

In [43]:
stereo_depth_fname = 'video/05_16_2025_10_33_14_trial_1/depth_data'

In [67]:
def natural_sort_key(s):
    """
    Sort strings containing numbers naturally (e.g., frame1.npy, frame2.npy, ..., frame10.npy)
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def load_depth_files(directory_path):
    """Load and sort all depth .npy files from the given directory"""
    if not os.path.exists(directory_path):
        print(f"Error: Directory '{directory_path}' not found.")
        return []
        
    # Find all .npy files in the directory
    depth_files = glob.glob(os.path.join(directory_path, "*.npy"))
    
    if not depth_files:
        print(f"Error: No .npy files found in '{directory_path}'")
        return []
    
    # Sort files naturally (so frame10 comes after frame9, not after frame1)
    depth_files.sort(key=natural_sort_key)
    print(f"Found {len(depth_files)} depth data files.")
    
    return depth_files


def visualize_depth_sequence(directory_path, start_idx=0, end_idx=None):
    """
    Interactive visualization with fixed figure for Jupyter notebooks
    """
    from ipywidgets import interact, interactive, IntSlider, Output
    import matplotlib.pyplot as plt
    from IPython.display import display
    
    # Try using widget backend for better interactivity
    try:
        %matplotlib widget
    except:
        %matplotlib inline
        print("Note: For better interactivity, run '%matplotlib widget' in a cell before calling this function")
    
    depth_files = load_depth_files(directory_path)
    if not depth_files:
        return
    
    if end_idx is None or end_idx > len(depth_files):
        end_idx = len(depth_files)
    
    depth_files = depth_files[start_idx:end_idx]
    
    # Create output widget for display
    out = Output()
    
    # Create the figure and initial plot
    with out:
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Load the first frame
        first_depth = np.load(depth_files[0])
        vmin = np.min(first_depth)
        vmax = np.max(first_depth)
        
        # Initial plot
        im = ax.imshow(first_depth, cmap='viridis', vmin=vmin, vmax=vmax)
        title = ax.set_title(f'Depth Data - Frame: {start_idx}')
        cbar = fig.colorbar(im, ax=ax, label='Disparity')
        # plt.tight_layout()
    
    # Update function with explicit display update
    def update_frame(frame_idx):
        with out:
            try:
                depth_data = np.load(depth_files[frame_idx])
                
                if np.isnan(depth_data).any() or np.isinf(depth_data).any():
                    depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=vmax, neginf=vmin)
                
                im.set_data(depth_data)
                title.set_text(f'Depth Data - Frame: {frame_idx + start_idx}')

                # Force redraw
                fig.canvas.draw()
                plt.draw()
            except Exception as e:
                print(f"Error loading frame {frame_idx}: {e}")
    
    # Create the interactive widget using interactive instead of interact
    slider = IntSlider(
        min=0,
        max=len(depth_files)-1,
        step=1,
        value=0,
        description='Frame:',
        continuous_update=True  # Try True to update while dragging
    )
    
    # Connect the widget to the update function
    interactive_plot = interactive(update_frame, frame_idx=slider)
    
    # Display both the output and the widget
    display(slider, out)
    
    # Initial update
    update_frame(0)
    
    return fig, ax

def create_depth_animation(directory_path, output_path=None, start_idx=0, end_idx=None, 
                          interval=100, dpi=100, fps=10):
    """
    Create an animation from a sequence of depth data .npy files
    
    Args:
        directory_path (str): Path to directory containing depth .npy files
        output_path (str): Path to save the animation (default: auto-generate)
        start_idx (int): Index of first frame to include
        end_idx (int): Index of last frame to include (None for all)
        interval (int): Delay between frames in milliseconds
        dpi (int): DPI for saved animation
        fps (int): Frames per second for saved animation
    """
    depth_files = load_depth_files(directory_path)
    if not depth_files:
        return
    
    if end_idx is None or end_idx > len(depth_files):
        end_idx = len(depth_files)
    
    depth_files = depth_files[start_idx:end_idx]
    
    # Generate output path if not provided
    if output_path is None:
        output_path = os.path.join(os.path.dirname(directory_path), 
                                   f"depth_animation_{start_idx}_{end_idx-1}.mp4")
    
    # Load the first file to get dimensions and create the plot
    try:
        first_depth = np.load(depth_files[0])
        
        # Set initial min/max values for the colormap
        vmin = np.min(first_depth)
        vmax = np.max(first_depth)
        
        # Handle NaN or inf values
        if np.isnan(first_depth).any() or np.isinf(first_depth).any():
            first_depth = np.nan_to_num(first_depth, nan=0.0, posinf=vmax, neginf=vmin)
    
    except Exception as e:
        print(f"Error loading first depth file: {e}")
        return
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(first_depth, cmap='viridis', vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax, label='Disparity')
    title = ax.set_title(f'Frame: {start_idx}')
    
    # Function to update the animation
    def update_frame(frame_idx):
        try:
            depth_data = np.load(depth_files[frame_idx])
            
            # Handle NaN or inf values
            if np.isnan(depth_data).any() or np.isinf(depth_data).any():
                depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=vmax, neginf=vmin)
            
            im.set_data(depth_data)
            title.set_text(f'Frame: {frame_idx + start_idx}')
            return [im, title]
        except Exception as e:
            print(f"Error in animation frame {frame_idx}: {e}")
            return [im, title]
    
    print(f"Creating animation with {len(depth_files)} frames...")
    ani = animation.FuncAnimation(fig, update_frame, frames=len(depth_files),
                                 interval=interval, blit=True)
    
    # Save the animation
    if output_path.endswith('.mp4'):
        writer = animation.FFMpegWriter(fps=fps)
        ani.save(output_path, writer=writer, dpi=dpi)
    else:
        ani.save(output_path, dpi=dpi, writer='pillow')
    
    print(f"Animation saved to {output_path}")
    plt.close()

def create_depth_montage(directory_path, output_path=None, num_frames=9, 
                         start_idx=0, end_idx=None, figsize=(15, 12)):
    """
    Create a montage of selected frames from depth data sequence
    
    Args:
        directory_path (str): Path to directory containing depth .npy files
        output_path (str): Path to save the montage (default: auto-generate)
        num_frames (int): Number of frames to include in montage
        start_idx (int): Index of first frame to include
        end_idx (int): Index of last frame to include (None for all)
        figsize (tuple): Figure size in inches
    """
    depth_files = load_depth_files(directory_path)
    if not depth_files:
        return
    
    if end_idx is None or end_idx > len(depth_files):
        end_idx = len(depth_files)
    
    depth_files = depth_files[start_idx:end_idx]
    
    # Generate output path if not provided
    if output_path is None:
        output_path = os.path.join(os.path.dirname(directory_path), 
                                   f"depth_montage_{start_idx}_{end_idx-1}.png")
    
    # Select frames evenly spaced throughout the sequence
    if num_frames > len(depth_files):
        num_frames = len(depth_files)
        selected_indices = list(range(len(depth_files)))
    else:
        selected_indices = [int(i * (len(depth_files) - 1) / (num_frames - 1)) for i in range(num_frames)]
    
    selected_files = [depth_files[i] for i in selected_indices]
    
    # Calculate grid dimensions
    grid_size = int(np.ceil(np.sqrt(num_frames)))
    
    # Create montage
    fig, axes = plt.subplots(grid_size, grid_size, figsize=figsize)
    axes = axes.flatten()
    
    # Load and plot each selected frame
    vmin = None
    vmax = None
    
    for i, (ax, file_path) in enumerate(zip(axes, selected_files)):
        if i >= num_frames:
            ax.axis('off')
            continue
            
        try:
            depth_data = np.load(file_path)
            
            # Set consistent colormap limits
            if vmin is None:
                vmin = np.min(depth_data)
            if vmax is None:
                vmax = np.max(depth_data)
            
            # Handle NaN or inf values
            if np.isnan(depth_data).any() or np.isinf(depth_data).any():
                depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=vmax, neginf=vmin)
            
            im = ax.imshow(depth_data, cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_title(f'Frame {selected_indices[i] + start_idx}')
            ax.axis('off')
        except Exception as e:
            print(f"Error loading frame {selected_indices[i]}: {e}")
            ax.axis('off')
    
    # Hide any unused subplots
    for i in range(num_frames, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    cbar = fig.colorbar(im, ax=axes, shrink=0.6, label='Disparity')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Montage saved to {output_path}")
    plt.show()
def convert_depth_npy_to_images(source_folder, output_folder=None, colormap=cv2.COLORMAP_VIRIDIS, 
                               quality=95, normalize=True, file_prefix="frame"):
    """
    Convert depth .npy files to jpg images with colormap visualization
    
    Args:
        source_folder (str): Path to folder containing depth .npy files
        output_folder (str): Path to folder where jpg images will be saved
        colormap (int): OpenCV colormap to use (default: COLORMAP_VIRIDIS)
        quality (int): JPEG quality (0-100)
        normalize (bool): Whether to normalize depth values across all frames
        file_prefix (str): Prefix for output filenames
    
    Returns:
        int: Number of files converted
    """
    # Check if source folder exists
    if not os.path.exists(source_folder):
        print(f"Error: Source directory '{source_folder}' not found.")
        return 0
    
    # Determine output folder if not specified
    if output_folder is None:
        # Get the parent directory of the source folder
        parent_dir = os.path.dirname(os.path.abspath(source_folder))
        output_folder = os.path.join(parent_dir, "depth")    
        
    # Find all .npy files in the source directory
    depth_files = glob.glob(os.path.join(source_folder, "*.npy"))
    
    if not depth_files:
        print(f"Error: No .npy files found in '{source_folder}'")
        return 0
    
    # Sort files naturally
    depth_files.sort(key=natural_sort_key)
    print(f"Found {len(depth_files)} depth data files.")
    
    # If we normalize globally, we need to find min/max values across all files
    global_min = float('inf')
    global_max = float('-inf')
    
    if normalize:
        print("Finding global min/max depth values...")
        for depth_file in tqdm(depth_files):
            try:
                depth_data = np.load(depth_file)
                
                # Skip NaN and inf values for min/max calculation
                valid_data = depth_data[~np.isnan(depth_data) & ~np.isinf(depth_data)]
                if len(valid_data) > 0:
                    file_min = np.min(valid_data)
                    file_max = np.max(valid_data)
                    
                    global_min = min(global_min, file_min)
                    global_max = max(global_max, file_max)
            except Exception as e:
                print(f"Error loading file {depth_file}: {e}")
        
        print(f"Global depth range: [{global_min:.4f}, {global_max:.4f}]")
        
        # If we couldn't find valid min/max (all files have errors or only contain NaN/inf)
        if global_min == float('inf') or global_max == float('-inf'):
            print("Warning: Could not determine valid global min/max. Using default 0-1 range.")
            global_min = 0
            global_max = 1
    
    # Process each depth file
    print("Converting depth files to images...")
    for i, depth_file in enumerate(tqdm(depth_files)):
        try:
            # Load depth data
            depth_data = np.load(depth_file)
            
            # Determine file-specific min/max if not normalizing globally
            if not normalize:
                valid_data = depth_data[~np.isnan(depth_data) & ~np.isinf(depth_data)]
                if len(valid_data) > 0:
                    depth_min = np.min(valid_data)
                    depth_max = np.max(valid_data)
                else:
                    depth_min = 0
                    depth_max = 1
            else:
                depth_min = global_min
                depth_max = global_max
            
            # Handle NaN or inf values
            depth_data = np.nan_to_num(depth_data, nan=depth_min, posinf=depth_max, neginf=depth_min)
            
            # Normalize to 0-255 range for 8-bit image
            depth_range = depth_max - depth_min
            if depth_range == 0:  # Avoid division by zero
                depth_range = 1
            
            normalized_depth = ((depth_data - depth_min) / depth_range * 255).astype(np.uint8)
            
            # Apply colormap
            colored_depth = cv2.applyColorMap(normalized_depth, colormap)
            
            # Extract base number from filename or use index
            try:
                # Try to extract the frame number from the filename (e.g., frame123.npy -> 123)
                file_basename = os.path.basename(depth_file)
                frame_num = int(re.search(r'(\d+)', file_basename).group(1))
            except (AttributeError, ValueError):
                # If extraction fails, use the index
                frame_num = i + 1
            
            # Save as JPG
            output_filename = f"{file_prefix}{frame_num}.jpg"
            output_path = os.path.join(output_folder, output_filename)
            cv2.imwrite(output_path, colored_depth, [cv2.IMWRITE_JPEG_QUALITY, quality])
            
        except Exception as e:
            print(f"Error processing {depth_file}: {e}")
    
    print(f"Converted {len(depth_files)} depth files to images in '{output_folder}'")
    return len(depth_files)

In [68]:
convert_depth_npy_to_images(stereo_depth_fname)

Found 2324 depth data files.
Finding global min/max depth values...


  0%|          | 0/2324 [00:00<?, ?it/s]

100%|██████████| 2324/2324 [00:01<00:00, 1186.00it/s]


Global depth range: [0.0000, 0.9922]
Converting depth files to images...


100%|██████████| 2324/2324 [00:06<00:00, 335.13it/s]

Converted 2324 depth files to images in '/home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/05_16_2025_10_33_14_trial_1/depth'


2324

In [45]:
create_depth_animation(stereo_depth_fname)

Found 2324 depth data files.
Creating animation with 2324 frames...
Animation saved to video/05_16_2025_10_33_14_trial_1/depth_animation_0_2323.mp4


In [ ]:
create_depth_montage(stereo_depth_fname)

In [ ]:
visualize_depth_sequence(stereo_depth_fname)

### Regular Stereo Footage

In [46]:
def create_video_from_images(image_folder, output_path, fps=30, extension="jpg"):
    """
    Create a video from a folder of images
    
    Args:
        image_folder (str): Path to folder containing images
        output_path (str): Path where to save the output video
        fps (int): Frames per second for the output video
        extension (str): File extension of the images to use
    
    Returns:
        bool: True if successful, False otherwise
    """
    # Check if folder exists
    if not os.path.exists(image_folder):
        print(f"Error: Directory '{image_folder}' not found.")
        return False
    
    # Find all image files in the directory
    image_files = glob.glob(os.path.join(image_folder, f"*.{extension}"))
    
    if not image_files:
        print(f"Error: No {extension} files found in '{image_folder}'")
        return False
    
    # Sort files naturally
    image_files.sort(key=natural_sort_key)
    print(f"Found {len(image_files)} images in '{image_folder}'")
    
    # Read first image to get dimensions
    first_image = cv2.imread(image_files[0])
    if first_image is None:
        print(f"Error: Could not read image '{image_files[0]}'")
        return False
    
    height, width, channels = first_image.shape
    
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'XVID' for .avi format
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Process each image
    print(f"Creating video: {output_path}")
    for image_file in tqdm(image_files):
        image = cv2.imread(image_file)
        if image is not None:
            video_writer.write(image)
    
    # Release the video writer
    video_writer.release()
    print(f"Video saved to {output_path}")
    
    return True

def create_stereo_videos(left_folder, right_folder, output_dir=None, fps=30, extension="jpg"):
    """
    Create separate videos from left and right image folders
    
    Args:
        left_folder (str): Path to folder containing left camera images
        right_folder (str): Path to folder containing right camera images
        output_dir (str): Directory to save output videos (default: current directory)
        fps (int): Frames per second for the output videos
        extension (str): File extension of the images to use
    """
    # Use current directory if output_dir is not specified
    if output_dir is None:
        output_dir = os.getcwd()
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Define output paths
    left_video_path = os.path.join(output_dir, "left_camera.mp4")
    right_video_path = os.path.join(output_dir, "right_camera.mp4")
    
    # Create videos
    left_success = create_video_from_images(left_folder, left_video_path, fps, extension)
    right_success = create_video_from_images(right_folder, right_video_path, fps, extension)
    
    # Check results
    if left_success and right_success:
        print("\nBoth videos created successfully:")
        print(f"Left camera video: {left_video_path}")
        print(f"Right camera video: {right_video_path}")
    else:
        print("\nThere were issues creating one or both videos.")

In [50]:
stereo_fname = 'video/05_16_2025_10_33_14_trial_1/'
# Create videos from image folders
create_stereo_videos(
    left_folder=stereo_fname + "left", 
    right_folder=stereo_fname + "right",
    output_dir=stereo_fname,
    fps=10,                             # Optional
    extension="jpg"                     # Optional
)

Found 2324 images in 'video/05_16_2025_10_33_14_trial_1/left'
Creating video: video/05_16_2025_10_33_14_trial_1/left_camera.mp4


100%|██████████| 2324/2324 [00:08<00:00, 280.04it/s]


Video saved to video/05_16_2025_10_33_14_trial_1/left_camera.mp4
Found 2324 images in 'video/05_16_2025_10_33_14_trial_1/right'
Creating video: video/05_16_2025_10_33_14_trial_1/right_camera.mp4


100%|██████████| 2324/2324 [00:08<00:00, 268.61it/s]


Video saved to video/05_16_2025_10_33_14_trial_1/right_camera.mp4

Both videos created successfully:
Left camera video: video/05_16_2025_10_33_14_trial_1/left_camera.mp4
Right camera video: video/05_16_2025_10_33_14_trial_1/right_camera.mp4


In [69]:
def create_composite_video(left_folder, right_folder, depth_folder, output_path, 
                         fps=30, extension="jpg", depth_extension="npy", start_frame=0):
    """
    Create a composite video showing left, right, and depth images with frame counter
    
    Args:
        left_folder (str): Path to folder containing left camera images
        right_folder (str): Path to folder containing right camera images
        depth_folder (str): Path to folder containing depth data files
        output_path (str): Path where to save the output video
        fps (int): Frames per second for the output video
        extension (str): File extension for left/right images
        depth_extension (str): File extension for depth files
        start_frame (int): Starting frame number for the counter
    
    Returns:
        bool: True if successful, False otherwise
    """
    # Check if folders exist
    if not all(os.path.exists(folder) for folder in [left_folder, right_folder, depth_folder]):
        print("Error: One or more directories not found.")
        return False
    
    # Find all files
    left_files = glob.glob(os.path.join(left_folder, f"*.{extension}"))
    right_files = glob.glob(os.path.join(right_folder, f"*.{extension}"))
    depth_files = glob.glob(os.path.join(depth_folder, f"*.{depth_extension}"))
    
    if not all([left_files, right_files, depth_files]):
        print("Error: No files found in one or more folders.")
        return False
    
    # Sort files naturally
    left_files.sort(key=natural_sort_key)
    right_files.sort(key=natural_sort_key)
    depth_files.sort(key=natural_sort_key)
    
    # Ensure all folders have the same number of files
    num_frames = min(len(left_files), len(right_files), len(depth_files))
    print(f"Using {num_frames} frames from each folder")
    
    left_files = left_files[:num_frames]
    right_files = right_files[:num_frames]
    depth_files = depth_files[:num_frames]
    
    # Read first images to get dimensions
    first_left = cv2.imread(left_files[0])
    first_right = cv2.imread(right_files[0])
    
    if first_left is None or first_right is None:
        print("Error: Could not read one or both of the first images")
        return False
    
    # Load first depth file to get dimensions
    try:
        first_depth = np.load(depth_files[0])
        # Normalize depth for visualization (convert to grayscale image)
        depth_min = np.min(first_depth[~np.isnan(first_depth) & ~np.isinf(first_depth)])
        depth_max = np.max(first_depth[~np.isnan(first_depth) & ~np.isinf(first_depth)])
        first_depth = np.nan_to_num(first_depth, nan=depth_min, posinf=depth_max, neginf=depth_min)
        normalized_depth = ((first_depth - depth_min) / (depth_max - depth_min) * 255).astype(np.uint8)
        # Convert to color image for visualization
        first_depth_color = cv2.applyColorMap(normalized_depth, cv2.COLORMAP_VIRIDIS)
    except Exception as e:
        print(f"Error loading first depth file: {e}")
        return False
    
    # Get dimensions
    h_left, w_left, _ = first_left.shape
    h_right, w_right, _ = first_right.shape
    h_depth, w_depth = first_depth.shape[:2]
    
    # Resize all images to the same height
    target_height = min(h_left, h_right, h_depth)
    
    # Calculate new widths maintaining aspect ratio
    w_left_new = int(w_left * target_height / h_left)
    w_right_new = int(w_right * target_height / h_right)
    w_depth_new = int(w_depth * target_height / h_depth)
    
    # Compute composite dimensions
    # We'll place images in a row: [Left | Right | Depth]
    composite_width = w_left_new + w_right_new + w_depth_new
    composite_height = target_height + 30  # Extra space for frame counter at the bottom
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (composite_width, composite_height))
    
    # Process each set of images
    print(f"Creating composite video: {output_path}")
    for i in tqdm(range(num_frames)):
        # Read left and right images
        left_img = cv2.imread(left_files[i])
        right_img = cv2.imread(right_files[i])
        
        # Load and process depth
        try:
            depth_data = np.load(depth_files[i])
            # Handle NaN or inf values
            depth_data = np.nan_to_num(depth_data, nan=depth_min, posinf=depth_max, neginf=depth_min)
            # Normalize to 0-255 for visualization
            normalized_depth = ((depth_data - depth_min) / (depth_max - depth_min) * 255).astype(np.uint8)
            # Apply colormap for better visualization
            depth_color = cv2.applyColorMap(normalized_depth, cv2.COLORMAP_VIRIDIS)
        except Exception as e:
            print(f"Error processing depth file {i}: {e}")
            # Create a blank image if there's an error
            depth_color = np.zeros((h_depth, w_depth, 3), dtype=np.uint8)
        
        # Resize images to target height
        left_resized = cv2.resize(left_img, (w_left_new, target_height))
        right_resized = cv2.resize(right_img, (w_right_new, target_height))
        depth_resized = cv2.resize(depth_color, (w_depth_new, target_height))
        
        # Create composite frame with black background
        composite = np.zeros((composite_height, composite_width, 3), dtype=np.uint8)
        
        # Place images side by side
        composite[:target_height, :w_left_new] = left_resized
        composite[:target_height, w_left_new:w_left_new+w_right_new] = right_resized
        composite[:target_height, w_left_new+w_right_new:] = depth_resized
        
        # Add labels
        cv2.putText(composite, "Left", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(composite, "Right", (w_left_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(composite, "Depth", (w_left_new + w_right_new + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Add frame counter
        frame_text = f"Frame: {i + start_frame}"
        text_size = cv2.getTextSize(frame_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
        text_x = (composite_width - text_size[0]) // 2
        cv2.putText(composite, frame_text, (text_x, composite_height - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Add to video
        video_writer.write(composite)
    
    # Release video writer
    video_writer.release()
    print(f"Composite video saved to {output_path}")
    
    return True

In [72]:
stereo_fname = 'video/05_16_2025_10_33_14_trial_1/'
left_path = stereo_fname + 'left'
right_path = stereo_fname + 'right'
depth_path = stereo_fname + 'depth_data'
output_path = stereo_fname + "composite_video.mp4"
fps = 20

create_composite_video(
    left_path, 
    right_path, 
    depth_path, 
    output_path,
    fps
)

Using 2324 frames from each folder
Creating composite video: video/05_16_2025_10_33_14_trial_1/composite_video.mp4


100%|██████████| 2324/2324 [00:38<00:00, 59.80it/s]

Composite video saved to video/05_16_2025_10_33_14_trial_1/composite_video.mp4


True

### Stereo Point with Depth Images

In [ ]:
## set the x and y points in the depth images acquired 
print(npz_file['stereo_point'].shape)